# Tarea 1 · Parte 1 — Recolección y limpieza de noticias peruanas

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

1. Configuración inicial: Registrar la fecha de descarga, el medio elegido (A, B o C) y las URLs base en la primera celda Markdown.

| Campo | Valor |
|---|---|
| Estudiante | Valeria Murga |
| Opción elegida | C – Comparada (La República + El Comercio) |
| Fecha de descarga | se registra en la celda de configuración |
| URLs base | https://larepublica.pe · https://elcomercio.pe |
| Fuente de datos | RSS y sitemaps oficiales (publicados en el robots.txt de cada diario) |

In [6]:
# ---------------- Importación ----------------
import os, re, time, html, hashlib
from datetime import datetime, date, timezone
from urllib import robotparser #Leer el robots.txt y comprobar que nos permiten descargar
from urllib.parse import urlparse
import xml.etree.ElementTree as ET  #Leer los RSS, que vienen en formato XML

import requests  #Descargar los feeds
import pandas as pd  #Armar y limpiar la tabla
from bs4 import BeautifulSoup #Limpiar el HTML que a veces viene dentro de los textos
import plotly.express as px #Los gráficos del EDA

print("Librerías listas ✓")

Librerías listas ✓


In [7]:
# ---------------- Configuración ----------------
OPCION = "C"    # "A" La República | "B" El Comercio | "C" Comparada
FECHA_EXTRACCION_UTC = datetime.now(timezone.utc).isoformat()  # momento exacto
FECHA_DESCARGA = date.today().isoformat() # solo la fecha (para nombres de archivo)
PAUSA_SEG = 1.5 # pausa entre solicitudes (1–2 s)
USER_AGENT = ("Mozilla/5.0 (compatible; TareaPUCP-QLAB/1.0; "
              "clasificacion de titulares; contacto: valeria.murga@pucp.edu.pe)")
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]

# Meta de noticias por medio y categoría (se recorta a las más recientes para balancear)
META_POR_CATEGORIA = {"A": 100, "B": 100, "C": 50}[OPCION]

DIR_DATOS = "datos"
DIR_RAW = os.path.join(DIR_DATOS, "raw")
os.makedirs(DIR_RAW, exist_ok=True)

print("Opción:", OPCION, "| Extracción (UTC):", FECHA_EXTRACCION_UTC, "| Meta por medio y categoría:", META_POR_CATEGORIA)

Opción: C | Extracción (UTC): 2026-10-17T22:56:12.615270+00:00 | Meta por medio y categoría: 50


2. Extracción: Extraer mínimo 350 noticias con las columnas: titular, bajada, medio, seccion_original, url, fecha. Usar RSS o HTML (requests + BeautifulSoup), implementando pausas entre peticiones y guardando respaldo crudo en datos/raw/.

    Fuentes: Cada fuente es un feed RSS o un sitemap de noticias de una sección. La sección del feed solo sirve para
saber dónde buscar; la `seccion_original` que se guarda se toma de la **ruta real de la URL** de cada nota
(por ejemplo `tecnologia/inteligencia-artificial`), y luego se mapea a las 4 categorías.

In [28]:
LR = "https://larepublica.pe"
EC = "https://elcomercio.pe"

FUENTES_LR = []
for sec in ["politica", "economia", "deportes", "ciencia"]:
    FUENTES_LR += [
        {"medio": "La Republica", "tipo": "rss",     "seccion_feed": sec, "url": f"{LR}/rss/{sec}.xml"},
        {"medio": "La Republica", "tipo": "sitemap", "seccion_feed": sec, "url": f"{LR}/sitemap/{sec}.xml"},
        {"medio": "La Republica", "tipo": "sitemap", "seccion_feed": sec, "url": f"{LR}/sitemap/web/{sec}.xml"},
    ]
# Sitemap general de noticias declarado en robots.txt (todas las secciones; el mapeo filtra las 4 categorías)
FUENTES_LR.append({"medio": "La Republica", "tipo": "sitemap", "seccion_feed": "general",
                   "url": f"{LR}/sitemap/noticias.xml"})
FUENTES_EC = []
for sec in ["politica", "economia", "deporte-total", "tecnologia"]:
    FUENTES_EC += [
        {"medio": "El Comercio", "tipo": "rss",     "seccion_feed": sec, "url": f"{EC}/arc/outboundfeeds/rss/category/{sec}/?outputType=xml"},
        {"medio": "El Comercio", "tipo": "rss",     "seccion_feed": sec, "url": f"{EC}/arc/outboundfeeds/google-news-feed/category/{sec}/?outputType=xml"},
        {"medio": "El Comercio", "tipo": "sitemap", "seccion_feed": sec, "url": f"{EC}/sitemap/news/{sec}/?outputType=xml"},
    ]

FUENTES = {"A": FUENTES_LR, "B": FUENTES_EC, "C": FUENTES_LR + FUENTES_EC}[OPCION]
pd.DataFrame(FUENTES)

,medio,tipo,seccion_feed,url
0,La Republica,rss,politica,https://larepublica.pe/rss/politica.xml
1,La Republica,sitemap,politica,https://larepublica.pe/sitemap/politica.xml
2,La Republica,sitemap,politica,https://larepublica.pe/sitemap/web/politica.xml
3,La Republica,rss,economia,https://larepublica.pe/rss/economia.xml
4,La Republica,sitemap,economia,https://larepublica.pe/sitemap/economia.xml
5,La Republica,sitemap,economia,https://larepublica.pe/sitemap/web/economia.xml
6,La Republica,rss,deportes,https://larepublica.pe/rss/deportes.xml
7,La Republica,sitemap,deportes,https://larepublica.pe/sitemap/deportes.xml
8,La Republica,sitemap,deportes,https://larepublica.pe/sitemap/web/deportes.xml
9,La Republica,rss,ciencia,https://larepublica.pe/rss/ciencia.xml


Descarga (robots.txt, User-Agent, pausas, 1 reintento, respaldo crudo)

In [29]:
sesion = requests.Session()
sesion.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "es-PE,es;q=0.9"})
_robots = {}
log_solicitudes = []

def permitido(url):
    """Consulta robots.txt del dominio (se cachea por dominio)."""
    base = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
    if base not in _robots:
        rp = robotparser.RobotFileParser()
        try:
            r = sesion.get(base + "/robots.txt", timeout=20)
            rp.parse(r.text.splitlines())
        except Exception:
            rp = None
        _robots[base] = rp
        time.sleep(PAUSA_SEG)
    rp = _robots[base]
    return True if rp is None else rp.can_fetch(USER_AGENT, url)

def descargar(url, guardar_como=None):
    """GET con pausa y como máximo 1 reintento. Devuelve texto o None."""
    if not permitido(url):
        log_solicitudes.append({"url": url, "estado": "bloqueado_por_robots"})
        return None
    for intento in range(2):                       # intento inicial + 1 reintento
        try:
            r = sesion.get(url, timeout=25)
            time.sleep(PAUSA_SEG)
            log_solicitudes.append({"url": url, "estado": r.status_code, "intento": intento + 1})
            if r.status_code == 200:
                if guardar_como:
                    with open(os.path.join(DIR_RAW, guardar_como), "w", encoding="utf-8") as f:
                        f.write(r.text)
                return r.text
        except requests.RequestException as e:
            log_solicitudes.append({"url": url, "estado": f"error: {type(e).__name__}", "intento": intento + 1})
            time.sleep(PAUSA_SEG)
    return None

Procesamiento de RSS y sitemaps: Se utiliza xml.etree para extraer información de los archivos XML, sin considerar los espacios de nombres (media:, news:, dc:), lo que permite emplear el mismo código para ambos diarios.

In [30]:
def _local(tag):
    return tag.rsplit("}", 1)[-1]

def _hijo(elem, nombre):
    for c in elem:
        if _local(c.tag) == nombre:
            return c
    return None

def _txt(elem):
    return html.unescape((elem.text or "").strip()) if elem is not None else ""

def limpiar_texto(t):
    t = BeautifulSoup(t or "", "html.parser").get_text(" ")
    return re.sub(r"\s+", " ", html.unescape(t)).strip()

def parsear_rss(xml_texto):
    filas = []
    raiz = ET.fromstring(xml_texto.encode("utf-8"))
    for item in raiz.iter():
        if _local(item.tag) != "item":
            continue
        filas.append({
            "titular": limpiar_texto(_txt(_hijo(item, "title"))),
            "bajada":  limpiar_texto(_txt(_hijo(item, "description"))),
            "url":     _txt(_hijo(item, "link")),
            "fecha":   _txt(_hijo(item, "pubDate")),
        })
    return filas

def parsear_sitemap(xml_texto):
    filas = []
    raiz = ET.fromstring(xml_texto.encode("utf-8"))
    for u in raiz.iter():
        if _local(u.tag) != "url":
            continue
        loc = _txt(_hijo(u, "loc"))
        news = _hijo(u, "news")
        titulo = _txt(_hijo(news, "title")) if news is not None else ""
        fecha = _txt(_hijo(news, "publication_date")) if news is not None else _txt(_hijo(u, "lastmod"))
        if titulo:                                   # sitemaps sin título no sirven
            filas.append({"titular": limpiar_texto(titulo), "bajada": "", "url": loc, "fecha": fecha})
    return filas

In [31]:
registros = []
for f in FUENTES:
    nombre = f"{f['medio'].replace(' ', '')}_{f['tipo']}_{f['seccion_feed']}_{hashlib.md5(f['url'].encode()).hexdigest()[:6]}_{FECHA_DESCARGA}.xml"
    texto = descargar(f["url"], guardar_como=nombre)
    if texto is None:
        print("✗ sin respuesta:", f["url"]); continue
    try:
        filas = parsear_rss(texto) if f["tipo"] == "rss" else parsear_sitemap(texto)
    except ET.ParseError as e:
        print("✗ XML inválido:", f["url"], e); continue
    for x in filas:
        x.update({"medio": f["medio"], "fuente": f["tipo"], "seccion_feed": f["seccion_feed"], "fecha_extraccion_utc": FECHA_EXTRACCION_UTC})
    registros += filas
    print(f"✓ {len(filas):>4} entradas  {f['medio']:<13} {f['tipo']:<8} {f['url']}")

nuevo = pd.DataFrame(registros)
print("\nEntradas descargadas en esta corrida:", len(nuevo))

✓   16 entradas  La Republica  rss      https://larepublica.pe/rss/politica.xml
✓   16 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/politica.xml
✓    0 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/web/politica.xml
✓    9 entradas  La Republica  rss      https://larepublica.pe/rss/economia.xml
✓    9 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/economia.xml
✓    0 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/web/economia.xml
✓   12 entradas  La Republica  rss      https://larepublica.pe/rss/deportes.xml
✓   12 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/deportes.xml
✓    0 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/web/deportes.xml
✓   10 entradas  La Republica  rss      https://larepublica.pe/rss/ciencia.xml
✓   10 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/ciencia.xml
✓    0 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/web/ciencia

Acumulación de noticias entre días: Los feeds RSS muestran únicamente las noticias más recientes. Por ello, en cada ejecución del notebook, las nuevas noticias se incorporan al archivo datos/raw/acumulado.csv. 

In [32]:
RUTA_ACUM = os.path.join(DIR_RAW, "acumulado.csv")
if os.path.exists(RUTA_ACUM):
    previo = pd.read_csv(RUTA_ACUM, encoding="utf-8-sig")
    acumulado = pd.concat([previo, nuevo], ignore_index=True)
else:
    acumulado = nuevo.copy()
acumulado = acumulado.drop_duplicates(subset=["url"], keep="first")
acumulado.to_csv(RUTA_ACUM, index=False, encoding="utf-8-sig")
print("Total acumulado (URLs únicas):", len(acumulado))
pd.DataFrame(log_solicitudes, columns=["url", "estado", "intento"])["estado"].value_counts()  # resumen de respuestas HTTP

Total acumulado (URLs únicas): 604


estado
200    25
Name: count, dtype: int64

3. Limpieza y normalización:

Eliminación de duplicados por URL y titulares vacíos.

In [33]:
from urllib.parse import urlparse   # si ya está en tus imports, borra esta línea
import re

def seccion_desde_url(url):
    """Devuelve la sección (y subsección si existe) a partir de la ruta de la URL."""
    partes = [p for p in urlparse(url).path.split("/") if p]   # trozos de la ruta, sin vacíos
    if not partes:
        return ""
    # Si el 2.º trozo no es un número (año) ni el final de la nota, es una subsección
    if len(partes) > 2 and not partes[1].isdigit() and not partes[1].endswith("-noticia"):
        return f"{partes[0]}/{partes[1]}"
    return partes[0]

def fecha_desde_url(url):
    """La República trae /AAAA/MM/DD/ en la URL: es la fecha real de publicación."""
    m = re.search(r"/(20\d{2})/(\d{2})/(\d{2})/", url)
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}" if m else None

def normalizar_fecha(f):
    """Convierte la fecha del feed (formatos distintos) a AAAA-MM-DD en hora de Lima."""
    f = pd.to_datetime(f, errors="coerce", utc=True, format="mixed")
    return f.tz_convert("America/Lima").strftime("%Y-%m-%d") if pd.notna(f) else None

df = acumulado.copy()                                   # trabajamos sobre una copia
df["seccion_original"] = df["url"].apply(seccion_desde_url)
df["fecha"] = [fecha_desde_url(u) or normalizar_fecha(f) for u, f in zip(df["url"], df["fecha"])]
df["titular"] = df["titular"].fillna("").str.strip()
df["bajada"]  = df["bajada"].fillna("").str.strip()

# Inspección antes de limpiar
print(df.shape)
df.info()
print("\nNulos por columna:\n", df.isna().sum())
print("URLs repetidas:", df["url"].duplicated().sum())
df["seccion_original"].value_counts().head(30)

(604, 9)
<class 'pandas.DataFrame'>
Index: 604 entries, 0 to 744
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   titular               604 non-null    str  
 1   bajada                604 non-null    str  
 2   url                   604 non-null    str  
 3   fecha                 604 non-null    str  
 4   medio                 604 non-null    str  
 5   fuente                604 non-null    str  
 6   seccion_feed          604 non-null    str  
 7   fecha_extraccion_utc  604 non-null    str  
 8   seccion_original      604 non-null    str  
dtypes: str(9)
memory usage: 47.2 KB

Nulos por columna:
 titular                 0
bajada                  0
url                     0
fecha                   0
medio                   0
fuente                  0
seccion_feed            0
fecha_extraccion_utc    0
seccion_original        0
dtype: int64
URLs repetidas: 0


seccion_original
sociedad                              53
economia/peru                         46
politica                              43
tecnologia/actualidad                 37
espectaculos                          28
politica/congreso                     27
deporte-total/futbol-peruano          24
politica/gobierno                     22
economia                              21
mundo                                 20
deporte-total/futbol-mundial          19
deporte-total                         19
tecnologia/inteligencia-artificial    19
economia/mercados                     18
latinos-en-usa                        15
deportes                              12
ciencia                               10
economia/dia-1                        10
tecnologia/ciencias                   10
economia/opinion                       9
politica/actualidad                    8
deporte-total/polideportivo            8
deporte-total/argentina                7
tecnologia/ciberseguridad              7

Mapeo explícito de seccion_original a las 4 categorías finales (politica, economia, deportes, ciencia_tecnologia), mostrando la tabla de equivalencias.

In [34]:
MAPEO = {
    # Secciones raíz (sirven para ambos diarios y para cualquier subsección)
    "politica":      "politica",
    "elecciones":    "politica",
    "economia":      "economia",
    "deportes":      "deportes",
    "deporte-total": "deportes",
    "ciencia":       "ciencia_tecnologia",
    "tecnologia":    "ciencia_tecnologia",
    "eldominical/ciencia": "ciencia_tecnologia",
    # Excepciones: subsecciones que NO son de las 4 categorías
    "tecnologia/videojuegos": None,   # entretenimiento → se descarta
    "tecnologia/streamers":   None,   # farándula digital → se descarta
}

def mapear(seccion):
    """Busca la sección completa; si no está, usa la sección raíz (antes de '/')."""
    if seccion in MAPEO:
        return MAPEO[seccion]
    raiz = seccion.split("/")[0]
    return MAPEO.get(raiz)            # None si la raíz tampoco está → se descartará

df["etiqueta"] = df["seccion_original"].apply(mapear)

# Tabla de equivalencias como evidencia: cada sección encontrada y su categoría final
tabla_mapeo = (df.groupby(["medio", "seccion_original"])
                 .size().rename("n").reset_index())
tabla_mapeo["categoria_final"] = tabla_mapeo["seccion_original"].apply(lambda s: mapear(s) or "DESCARTADA")
pd.set_option("display.max_rows", 200)          # para que muestre todas las filas
tabla_mapeo.sort_values(["categoria_final", "n"], ascending=[True, False])

,medio,seccion_original,n,categoria_final
62,La Republica,sociedad,53,DESCARTADA
53,La Republica,espectaculos,28,DESCARTADA
57,La Republica,mundo,20,DESCARTADA
55,La Republica,latinos-en-usa,15,DESCARTADA
43,El Comercio,tecnologia/videojuegos,5,DESCARTADA
46,La Republica,cine-series,5,DESCARTADA
48,La Republica,cultural,5,DESCARTADA
59,La Republica,opinion,5,DESCARTADA
63,La Republica,verificador,5,DESCARTADA
14,El Comercio,ecdata,4,DESCARTADA


Reporte del número de noticias descartadas y motivos.
**Objetivo:** quedarnos solo con noticias válidas y dejar registro de todo lo que se eliminó y por qué.
**Qué hace:** aplica filtros en orden. Cada filtro guarda las filas eliminadas con su **motivo**:
1. Titular vacío o muy corto (menos de 4 palabras).
2. URL duplicada.
3. Titular duplicado (la misma nota publicada en dos URLs).
4. Sección fuera de las 4 categorías (según la tabla del Paso 6).
5. Formatos no textuales (videos, en vivo, galerías).

Después revisa el **balance** por medio y categoría frente a la meta de 50 de la opción C. La meta se usa como
**mínimo a revisar** y no como tope: si se recortara a 50, el total quedaría por debajo de las 350 noticias exigidas.
**Resultado esperado:** la tabla de descartes por motivo y la tabla de noticias finales por categoría y medio.

In [35]:
descartes = []                     # aquí se guardan las filas eliminadas, con su motivo

def registrar(mask, motivo):
    """Elimina de df las filas donde mask es True y las guarda en 'descartes'."""
    global df
    sub = df[mask].copy()
    sub["motivo"] = motivo
    descartes.append(sub)
    df = df[~mask].copy()
    print(f"{motivo:<50} → {int(mask.sum())}")

n_inicial = len(df)

registrar(df["titular"].str.split().str.len().fillna(0) < 4, "Titular vacío o muy corto (< 4 palabras)")
registrar(df.duplicated(subset=["url"]), "URL duplicada")

# Titular "normalizado": minúsculas y sin signos, para detectar la misma nota en dos URLs
clave = df["titular"].str.lower().str.replace(r"[^\wáéíóúñü ]", "", regex=True).str.strip()
registrar(clave.duplicated(), "Titular duplicado (misma nota en varias URLs)")

registrar(df["etiqueta"].isna(), "Sección fuera de las 4 categorías")
registrar(df["url"].str.contains(r"/(?:envivo|en-vivo|video|videos|galeria|fotogaleria)/", regex=True),
          "Formato no textual (video / en vivo / galería)")

df_desc = pd.concat(descartes, ignore_index=True)
print(f"\nInicial: {n_inicial} | Descartadas: {len(df_desc)} | Finales: {len(df)}")
df_desc.groupby("motivo").size().rename("noticias_descartadas").to_frame()

Titular vacío o muy corto (< 4 palabras)           → 2
URL duplicada                                      → 0
Titular duplicado (misma nota en varias URLs)      → 0
Sección fuera de las 4 categorías                  → 169
Formato no textual (video / en vivo / galería)     → 0

Inicial: 604 | Descartadas: 171 | Finales: 433


,noticias_descartadas
motivo,
Sección fuera de las 4 categorías,169
Titular vacío o muy corto (< 4 palabras),2


In [36]:
# Noticias finales por categoría y medio, y celdas que no llegan a la meta de la opción C
balance = pd.crosstab(df["etiqueta"], df["medio"], margins=True, margins_name="Total")
print("Celdas (medio × categoría) por debajo de la meta de", META_POR_CATEGORIA, ":")
print(balance.drop(index="Total", columns="Total").lt(META_POR_CATEGORIA).stack().loc[lambda s: s])
balance

Celdas (medio × categoría) por debajo de la meta de 50 :
etiqueta            medio       
ciencia_tecnologia  La Republica    True
deportes            La Republica    True
economia            La Republica    True
politica            La Republica    True
dtype: bool


medio,El Comercio,La Republica,Total
etiqueta,,,
ciencia_tecnologia,90,10,100
deportes,98,12,110
economia,99,9,108
politica,95,20,115
Total,382,51,433


Buscar más fuentes de La República en robots.txt

**Objetivo:** reducir el desbalance entre medios (La República aporta solo 51 notas).
**Qué hace:** lee el `robots.txt` de La República y lista los sitemaps que el propio diario declara.
**Resultado esperado:** la lista de sitemaps oficiales de La República.

In [27]:
texto = descargar(f"{LR}/robots.txt")
if texto:
    sitemaps = [l.split(":", 1)[1].strip() for l in texto.splitlines() if l.lower().startswith("sitemap:")]
    print(len(sitemaps), "sitemaps declarados:")
    for s in sitemaps:
        print("  ", s)
else:
    print("No se pudo leer robots.txt (revisa log_solicitudes)")

45 sitemaps declarados:
   https://larepublica.pe/sitemap/sociedad.xml
   https://larepublica.pe/sitemap/noticias.xml
   https://larepublica.pe/sitemap/politica.xml
   https://larepublica.pe/sitemap/opinion.xml
   https://larepublica.pe/sitemap/mundo.xml
   https://larepublica.pe/sitemap/mastergrama.xml
   https://larepublica.pe/sitemap/espectaculos.xml
   https://larepublica.pe/sitemap/economia.xml
   https://larepublica.pe/sitemap/deportes.xml
   https://larepublica.pe/sitemap/cine-series.xml
   https://larepublica.pe/sitemap/carlincatura.xml
   https://larepublica.pe/sitemap/ciencia.xml
   https://larepublica.pe/sitemap/horoscopo.xml
   https://larepublica.pe/sitemap/servicios.xml
   https://larepublica.pe/sitemap/latinos-en-usa.xml
   https://larepublica.pe/sitemap/web/horoscopo.xml
   https://larepublica.pe/sitemap/web/sociedad.xml
   https://larepublica.pe/sitemap/web/politica.xml
   https://larepublica.pe/sitemap/web/opinion.xml
   https://larepublica.pe/sitemap/web/servicios.xm

Se procedió a modificar desde el paso 2
**Balance por medio (limitación documentada):** El Comercio aporta 382 noticias y La República 51.
Para La República se probaron todos sus canales oficiales declarados en robots.txt: RSS por sección, sitemaps por
sección, sitemaps web y el sitemap general de noticias (que sumó 153 URLs, pero de otras secciones como sociedad,
mundo o espectáculos, y por eso se descartaron en el mapeo). Sus feeds solo exponen entre 9 y 20 notas por sección,
así que no alcanza la meta de 50 por categoría. Las 4 categorías sí quedan balanceadas en el total (100–115),
que es lo relevante para entrenar el clasificador. La comparación entre medios se interpreta con cautela.

### Completar bajadas faltantes
**Objetivo:** que cada noticia tenga su bajada (subtítulo), una de las columnas obligatorias.
**Qué hace:** los sitemaps traen solo el titular. Para las notas **finales** sin bajada se visita su página
(respetando robots.txt y la pausa de 1.5 s) y se lee la descripción oficial del diario
(`og:description`, `description` o `twitter:description`) con BeautifulSoup.
El resultado se guarda en `acumulado.csv` cada 25 notas, para no repetir solicitudes si se vuelve a ejecutar.
**Resultado esperado:** casi todas las noticias con bajada. Las que no la tengan se conservan,
porque el modelo se entrena con el titular.

In [37]:
from bs4 import BeautifulSoup     # si ya está en tus imports, borra esta línea

def bajada_desde_html(url):
    """Visita la nota y devuelve la descripción que publica el diario en sus etiquetas <meta>."""
    texto = descargar(url)
    if not texto:
        return ""
    soup = BeautifulSoup(texto, "html.parser")
    for attrs in ({"property": "og:description"}, {"name": "description"}, {"name": "twitter:description"}):
        m = soup.find("meta", attrs=attrs)
        if m and m.get("content"):
            return " ".join(m["content"].split())      # quita saltos de línea y espacios dobles
    return ""

faltan = df.index[df["bajada"].str.len() == 0]
print("Notas finales sin bajada:", len(faltan), f"(≈{len(faltan) * PAUSA_SEG / 60:.0f} min o más)")

for k, i in enumerate(faltan, start=1):
    b = bajada_desde_html(df.at[i, "url"])
    df.at[i, "bajada"] = b
    acumulado.loc[acumulado["url"] == df.at[i, "url"], "bajada"] = b   # también en el acumulado
    if k % 25 == 0:                                                     # guardado parcial cada 25
        acumulado.to_csv(RUTA_ACUM, index=False, encoding="utf-8-sig")
        print(f"  {k}/{len(faltan)} procesadas")

acumulado.to_csv(RUTA_ACUM, index=False, encoding="utf-8-sig")
print("Siguen sin bajada:", int((df["bajada"].str.len() == 0).sum()))

Notas finales sin bajada: 235 (≈6 min o más)
  25/235 procesadas
  50/235 procesadas
  75/235 procesadas
  100/235 procesadas
  125/235 procesadas
  150/235 procesadas
  175/235 procesadas
  200/235 procesadas
  225/235 procesadas
Siguen sin bajada: 0
